# 개별종목 조합I — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4146,0.5012,-0.0866,0.3611,0.3616,0.0449,0.3498,0.2679,0.3366
1,2,balanced,980,20150123,20150421,0.3680,0.3978,-0.0298,0.3529,0.3549,0.0370,0.3533,0.2847,0.3310
2,3,NaN,1210,20151228,20160328,0.3710,0.3762,-0.0051,0.3670,0.3671,0.0520,0.3603,0.3521,0.3632
3,4,NaN,1439,20161202,20170228,0.3845,0.4617,-0.0772,0.3269,0.3344,0.0059,0.3551,0.2332,0.3015
4,5,balanced,1669,20171113,20180207,0.3732,0.3901,-0.0169,0.3611,0.3624,0.0456,0.3671,0.3060,0.3441
5,6,balanced,1899,20181024,20190118,0.3667,0.3725,-0.0058,0.3667,0.3692,0.0552,0.3804,0.3975,0.3764
6,7,balanced,2129,20190930,20191224,0.4112,0.4781,-0.0669,0.3583,0.3633,0.0540,0.3692,0.2962,0.3489
7,8,balanced,2359,20200902,20201130,0.3769,0.3476,0.0293,0.3767,0.3807,0.0702,0.3651,0.4257,0.3918
8,9,balanced,2589,20210806,20211105,0.3619,0.3914,-0.0295,0.3530,0.3585,0.0349,0.3630,0.3140,0.3416
9,10,NaN,2818,20220714,20221012,0.3565,0.3454,0.0110,0.3512,0.3546,0.0315,0.3540,0.3061,0.3364


,OOS 폴드 평균
accuracy,0.3744
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0225
macro_f1,0.3558
balanced_accuracy,0.3589
mcc,0.0403
pr_auc_macro_ovr,0.3617
down_recall,0.3205
core_harmonic_mean,0.3464


재실행 명령: python scripts/run_stock_model_experiment.py
